# Using llama with pydantic

Example on how to use llama_extract with Pydantic. Original example obtained from: [run-llama /llama-extract](https://github.com/run-llama/llama_extract/blob/main/examples/demo_pydantic_model.ipynb)

[LLAMA 3.1 community license agreement](https://llama.meta.com/llama3_1/license/)

In [1]:
import os
from pydantic import BaseModel
from llama_extract import LlamaExtract
import nest_asyncio

In [2]:
nest_asyncio.apply()

In [3]:
fnames = os.listdir(DATA_DIR)
fnames = [fname for fname in fnames if fname.endswith(".pdf")]
fpaths = [os.path.join(DATA_DIR, fname) for fname in fnames]
fpaths

In [4]:
DATA_DIR = "../../Data/testdata/resumes"

In [5]:
fnames = os.listdir(DATA_DIR)
fnames = [fname for fname in fnames if fname.endswith(".pdf")]
fpaths = [os.path.join(DATA_DIR, fname) for fname in fnames]
fpaths

['../../Data/testdata/resumes/14224370.pdf',
 '../../Data/testdata/resumes/12780508.pdf',
 '../../Data/testdata/resumes/19545827.pdf']

In [6]:
fpaths[0]

'../../Data/testdata/resumes/14224370.pdf'

In [7]:
class Education(BaseModel):
    degree: str
    honors: str
    institution: str
    field_of_study: str
    graudation_year: str


class Resume(BaseModel):
    education: Education
    summary: str

In [8]:
Resume.schema()

{'$defs': {'Education': {'properties': {'degree': {'title': 'Degree',
     'type': 'string'},
    'honors': {'title': 'Honors', 'type': 'string'},
    'institution': {'title': 'Institution', 'type': 'string'},
    'field_of_study': {'title': 'Field Of Study', 'type': 'string'},
    'graudation_year': {'title': 'Graudation Year', 'type': 'string'}},
   'required': ['degree',
    'honors',
    'institution',
    'field_of_study',
    'graudation_year'],
   'title': 'Education',
   'type': 'object'}},
 'properties': {'education': {'$ref': '#/$defs/Education'},
  'summary': {'title': 'Summary', 'type': 'string'}},
 'required': ['education', 'summary'],
 'title': 'Resume',
 'type': 'object'}

In [9]:
extractor = LlamaExtract(verbose=True)

In [10]:
#schema_response = await extractor.acreate_schema("Resume Schema", data_schema=Resume)
extraction_schema = await extractor.ainfer_schema("Test Schema", fpaths)
#

ApiError: status_code: 500, body: {'detail': 'Oops! Something went wrong on our end: Error reading parsed file from S3. Please try again in a few minutes. If the problem persists, please contact support by clicking the chat icon on cloud.llamaindex.ai providing this correlation ID: ac90398b-bd08-4aea-8809-9e562fdf7885'}

In [33]:
extractor.list_schemas()

[ExtractionSchema(id='d67d98bb-f698-458e-ba15-d84927d225c1', created_at=datetime.datetime(2024, 7, 29, 13, 58, 29, 450419, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2024, 7, 29, 13, 58, 29, 450419, tzinfo=datetime.timezone.utc), name='TEST_SCHEMA_2', project_id='8d99223b-8850-4482-817d-d15547d9f765', data_schema={'type': 'object', 'properties': {'Skills': {'type': 'array', 'items': {'type': 'string'}}, 'Education': {'type': 'object', 'properties': {'degree': {'type': 'string'}, 'institution': {'type': 'string'}, 'fieldOfStudy': {'type': 'string'}, 'graduationYear': {'type': 'string'}}}, 'Accountant': {'type': 'object', 'properties': {'skills': {'type': 'array', 'items': {'type': 'string'}}, 'summary': {'type': 'string'}, 'education': {'type': 'object', 'properties': {'degree': {'type': 'string'}, 'honors': {'type': 'string'}, 'institution': {'type': 'string'}, 'fieldOfStudy': {'type': 'string'}, 'graduationYear': {'type': 'string'}}}, 'experience': {'type': 'array', '

In [34]:
schema_response = await extractor.acreate_schema("Resume Schema", data_schema=extraction_schema.schema())

<frozen abc>:123: RuntimeWarning: coroutine 'LlamaExtract.alist_schemas' was never awaited


ApiError: status_code: 500, body: {'detail': 'Oops! Something went wrong on our end. Please try again in a few minutes. If the problem persists, please contact support by clicking the chat icon on cloud.llamaindex.ai providing this correlation ID: 7cf66c56-b4d8-40ec-8bce-44e0d20a0838'}

In [24]:
extraction_schema.data_schema

{'type': 'object',
 '$schema': 'http://json-schema.org/draft-07/schema#',
 'properties': {'skills': {'type': 'array', 'items': {'type': 'string'}},
  'summary': {'type': 'string'},
  'education': {'type': 'object',
   'properties': {'gpa': {'type': 'string'},
    'major': {'type': 'string'},
    'degree': {'type': 'string'},
    'honors': {'type': 'string'},
    'location': {'type': 'string'},
    'institution': {'type': 'string'},
    'graduationDate': {'type': 'string', 'format': 'date'},
    'completionHours': {'type': 'integer'}}},
  'experience': {'type': 'array',
   'items': {'type': 'object',
    'properties': {'endDate': {'type': 'string', 'format': 'date'},
     'jobTitle': {'type': 'string'},
     'location': {'type': 'string'},
     'startDate': {'type': 'string', 'format': 'date'},
     'companyName': {'type': 'string'},
     'responsibilities': {'type': 'array', 'items': {'type': 'string'}}}}},
  'highlights': {'type': 'array', 'items': {'type': 'string'}},
  'supervision'

In [37]:
result = await extractor.aextract(extraction_schema.id, fpaths)

Extracting files: 100%|██████████| 3/3 [00:50<00:00, 16.75s/it]


In [49]:
result[0].data['accountant']['experience'][0]['responsibilities']

['Prepare federal tax returns for individuals and small businesses.',
 'Perform bookkeeping and prepare financial statements for small businesses.',
 'Perform special projects & short-term assignments such as accountant at MCT Sheet Metal, Inc.']

In [20]:
{'type': 'object',
 '$defs': 
      {'Education': {
            'type': 'object',
            'title': 'Education',
            'required': 
                  ['degree',
                  'honors',
                  'institution',
                  'field_of_study',
                  'graudation_year'],
            'properties': {
                  'degree': {'type': 'string', 'title': 'Degree'},
                  'honors': {'type': 'string', 'title': 'Honors'},
                  'institution': {'type': 'string', 'title': 'Institution'},
                  'field_of_study': {'type': 'string', 'title': 'Field Of Study'},
                  'graudation_year': {'type': 'string', 'title': 'Graudation Year'}
                  }
            }
      },
'title': 'Resume',
      'required': [
            'education',
            'summary'],
      'properties': 
            {
            'summary': {'type': 'string', 'title': 'Summary'},
            'education': {'$ref': '#/$defs/Education'}
            }
}

{'type': 'object',
 '$defs': {'Education': {'type': 'object',
   'title': 'Education',
   'required': ['degree',
    'honors',
    'institution',
    'field_of_study',
    'graudation_year'],
   'properties': {'degree': {'type': 'string', 'title': 'Degree'},
    'honors': {'type': 'string', 'title': 'Honors'},
    'institution': {'type': 'string', 'title': 'Institution'},
    'field_of_study': {'type': 'string', 'title': 'Field Of Study'},
    'graudation_year': {'type': 'string', 'title': 'Graudation Year'}}}},
 'title': 'Resume',
 'required': ['education', 'summary'],
 'properties': {'summary': {'type': 'string', 'title': 'Summary'},
  'education': {'$ref': '#/$defs/Education'}}}